# B1

In [1]:
import gmsh

gmsh.initialize()

gmsh.model.add("glacier_terminus")

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 1500.0
Ly = 750.0
Lz = 125.0

# -------------------------------------------------------------------------
# Notch dimensions [m]
# -------------------------------------------------------------------------

lx = 5.0
ly = 10.0
lz = 10.0

# Spacing between notch centers [m]
s = 90.0

# -------------------------------------------------------------------------
# Mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 1.0
h_max = 18.0

# Local crack-tip refinement
h_refined = 2.5

refinement_x = 10.0
refinement_y = 10.0
refinement_z = 10.0

transition_thickness = 10.0

gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)
gmsh.option.setNumber("Mesh.MeshSizeFactor", 1.0)

# -------------------------------------------------------------------------
# Glacier geometry
# -------------------------------------------------------------------------

glacier = gmsh.model.occ.addBox(
    0.0,
    0.0,
    0.0,
    Lx,
    Ly,
    Lz,
)

# -------------------------------------------------------------------------
# 15 equally spaced notch locations
#
# 15 cracks on y = 0
# 15 cracks on y = Ly
#
# Total = 30 cracks
#
# The middle crack is at x = Lx / 2.
# With 15 cracks, indices run from -7 to +7:
#
# x = Lx/2 + i*s,  i = -7, ..., 0, ..., +7
# -------------------------------------------------------------------------

x_center = 0.5 * Lx

notch_centers = [
    x_center + i * s
    for i in range(-7, 8)
]

notches = []

for xc in notch_centers:

    # ---------------------------------------------------------------------
    # Notch on y = 0 side
    # ---------------------------------------------------------------------

    notch_bottom = gmsh.model.occ.addBox(
        xc - 0.5 * lx,
        0.0,
        Lz - lz,
        lx,
        ly,
        lz,
    )

    notches.append(
        (3, notch_bottom)
    )

    # ---------------------------------------------------------------------
    # Opposite notch on y = Ly side
    # ---------------------------------------------------------------------

    notch_top = gmsh.model.occ.addBox(
        xc - 0.5 * lx,
        Ly - ly,
        Lz - lz,
        lx,
        ly,
        lz,
    )

    notches.append(
        (3, notch_top)
    )

# -------------------------------------------------------------------------
# Subtract all 30 notches
# -------------------------------------------------------------------------

domain, _ = gmsh.model.occ.cut(
    [(3, glacier)],
    notches,
    removeObject=True,
    removeTool=True,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Internal planes
#
# z = Lz / 4 = 31.25 m
# z = Lz / 2 = 62.50 m
# -------------------------------------------------------------------------

z_quarter = Lz / 4.0
z_half = Lz / 2.0

plane_quarter = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z_quarter,
    Lx,
    Ly,
)

plane_half = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z_half,
    Lx,
    Ly,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Fragment glacier with internal planes
# -------------------------------------------------------------------------

domain, _ = gmsh.model.occ.fragment(
    domain,
    [
        (2, plane_quarter),
        (2, plane_half),
    ],
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Physical volume
# -------------------------------------------------------------------------

volume_tags = [
    tag
    for dim, tag in domain
    if dim == 3
]

gmsh.model.addPhysicalGroup(
    3,
    volume_tags,
    1,
)

gmsh.model.setPhysicalName(
    3,
    1,
    "GLACIER",
)

# -------------------------------------------------------------------------
# Local refinement around all 30 crack tips
# -------------------------------------------------------------------------

refinement_fields = []

crack_tip_z = Lz - lz

for xc in notch_centers:

    # ---------------------------------------------------------------------
    # Crack tip on y = 0 side
    # ---------------------------------------------------------------------

    crack_tip_y_bottom = ly

    field_bottom = gmsh.model.mesh.field.add("Box")

    gmsh.model.mesh.field.setNumber(
        field_bottom,
        "VIn",
        h_refined,
    )

    gmsh.model.mesh.field.setNumber(
        field_bottom,
        "VOut",
        h_max,
    )

    gmsh.model.mesh.field.setNumber(
        field_bottom,
        "XMin",
        xc - refinement_x,
    )

    gmsh.model.mesh.field.setNumber(
        field_bottom,
        "XMax",
        xc + refinement_x,
    )

    gmsh.model.mesh.field.setNumber(
        field_bottom,
        "YMin",
        max(
            0.0,
            crack_tip_y_bottom - refinement_y,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_bottom,
        "YMax",
        min(
            Ly,
            crack_tip_y_bottom + refinement_y,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_bottom,
        "ZMin",
        max(
            0.0,
            crack_tip_z - refinement_z,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_bottom,
        "ZMax",
        Lz,
    )

    gmsh.model.mesh.field.setNumber(
        field_bottom,
        "Thickness",
        transition_thickness,
    )

    refinement_fields.append(
        field_bottom
    )

    # ---------------------------------------------------------------------
    # Crack tip on y = Ly side
    # ---------------------------------------------------------------------

    crack_tip_y_top = Ly - ly

    field_top = gmsh.model.mesh.field.add("Box")

    gmsh.model.mesh.field.setNumber(
        field_top,
        "VIn",
        h_refined,
    )

    gmsh.model.mesh.field.setNumber(
        field_top,
        "VOut",
        h_max,
    )

    gmsh.model.mesh.field.setNumber(
        field_top,
        "XMin",
        xc - refinement_x,
    )

    gmsh.model.mesh.field.setNumber(
        field_top,
        "XMax",
        xc + refinement_x,
    )

    gmsh.model.mesh.field.setNumber(
        field_top,
        "YMin",
        max(
            0.0,
            crack_tip_y_top - refinement_y,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_top,
        "YMax",
        min(
            Ly,
            crack_tip_y_top + refinement_y,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_top,
        "ZMin",
        max(
            0.0,
            crack_tip_z - refinement_z,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_top,
        "ZMax",
        Lz,
    )

    gmsh.model.mesh.field.setNumber(
        field_top,
        "Thickness",
        transition_thickness,
    )

    refinement_fields.append(
        field_top
    )

# -------------------------------------------------------------------------
# Combine all 30 refinement regions
# -------------------------------------------------------------------------

field_min = gmsh.model.mesh.field.add("Min")

gmsh.model.mesh.field.setNumbers(
    field_min,
    "FieldsList",
    refinement_fields,
)

gmsh.model.mesh.field.setAsBackgroundMesh(
    field_min
)

# -------------------------------------------------------------------------
# Mesh settings
# -------------------------------------------------------------------------

gmsh.option.setNumber(
    "Mesh.Algorithm3D",
    10,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeFromCurvature",
    0,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeFromPoints",
    0,
)

gmsh.option.setNumber(
    "Mesh.MeshSizeExtendFromBoundary",
    0,
)

# -------------------------------------------------------------------------
# Generate tetrahedral mesh
# -------------------------------------------------------------------------

gmsh.model.mesh.generate(3)

# -------------------------------------------------------------------------
# Output
# -------------------------------------------------------------------------

gmsh.write("11/Lx1500_30C_S90.msh")

gmsh.finalize()

Info    : Meshing 1D...nts                                                                                                                
Info    : [  0%] Meshing curve 414 (Line)
Info    : [ 10%] Meshing curve 415 (Line)
Info    : [ 10%] Meshing curve 416 (Line)
Info    : [ 10%] Meshing curve 417 (Line)
Info    : [ 10%] Meshing curve 418 (Line)
Info    : [ 10%] Meshing curve 419 (Line)
Info    : [ 10%] Meshing curve 420 (Line)
Info    : [ 10%] Meshing curve 421 (Line)
Info    : [ 10%] Meshing curve 422 (Line)
Info    : [ 10%] Meshing curve 423 (Line)
Info    : [ 10%] Meshing curve 424 (Line)
Info    : [ 10%] Meshing curve 425 (Line)
Info    : [ 10%] Meshing curve 426 (Line)
Info    : [ 10%] Meshing curve 427 (Line)
Info    : [ 10%] Meshing curve 428 (Line)
Info    : [ 10%] Meshing curve 429 (Line)
Info    : [ 10%] Meshing curve 430 (Line)
Info    : [ 10%] Meshing curve 431 (Line)
Info    : [ 10%] Meshing curve 432 (Line)
Info    : [ 10%] Meshing curve 433 (Line)
Info    : [ 10%] Mesh

# XDMF - B1

In [2]:
import meshio

mesh_files = ["11/Lx1500_30C_S90.msh"]

for mesh_file in mesh_files:
    mesh = meshio.read(mesh_file)
    cells = mesh.get_cells_type("tetra")
    points = mesh.points

    meshio.write(
        mesh_file.replace(".msh", ".xdmf"),
        meshio.Mesh(
            points=points,
            cells={"tetra": cells},
        ),
    )
   

# Outline

In [3]:
import gmsh
import os

gmsh.initialize()

gmsh.model.add("glacier_terminus_30C_S90_outline")

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 1500.0
Ly = 750.0
Lz = 125.0

# -------------------------------------------------------------------------
# Notch dimensions [m]
# -------------------------------------------------------------------------

lx = 5.0
ly = 10.0
lz = 10.0

# Spacing between notch centers [m]
s = 90.0

# -------------------------------------------------------------------------
# Outline mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 1.0
h_max = 18.0

gmsh.option.setNumber("Mesh.MeshSizeMin", h_min)
gmsh.option.setNumber("Mesh.MeshSizeMax", h_max)
gmsh.option.setNumber("Mesh.MeshSizeFactor", 1.0)

# -------------------------------------------------------------------------
# Glacier geometry
# -------------------------------------------------------------------------

glacier = gmsh.model.occ.addBox(
    0.0,
    0.0,
    0.0,
    Lx,
    Ly,
    Lz,
)

# -------------------------------------------------------------------------
# 15 equally spaced notch locations
#
# 15 cracks on y = 0
# 15 cracks on y = Ly
#
# Total = 30 cracks
# -------------------------------------------------------------------------

x_center = 0.5 * Lx

notch_centers = [
    x_center + i * s
    for i in range(-7, 8)
]

notches = []

for xc in notch_centers:

    # ---------------------------------------------------------------------
    # Notch on y = 0 side
    # ---------------------------------------------------------------------

    notch_bottom = gmsh.model.occ.addBox(
        xc - 0.5 * lx,
        0.0,
        Lz - lz,
        lx,
        ly,
        lz,
    )

    notches.append(
        (3, notch_bottom)
    )

    # ---------------------------------------------------------------------
    # Notch on y = Ly side
    # ---------------------------------------------------------------------

    notch_top = gmsh.model.occ.addBox(
        xc - 0.5 * lx,
        Ly - ly,
        Lz - lz,
        lx,
        ly,
        lz,
    )

    notches.append(
        (3, notch_top)
    )

# -------------------------------------------------------------------------
# Subtract all 30 notches
# -------------------------------------------------------------------------

domain, _ = gmsh.model.occ.cut(
    [(3, glacier)],
    notches,
    removeObject=True,
    removeTool=True,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Internal planes
#
# z = Lz / 4
# z = Lz / 2
# -------------------------------------------------------------------------

z_quarter = Lz / 4.0
z_half = Lz / 2.0

plane_quarter = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z_quarter,
    Lx,
    Ly,
)

plane_half = gmsh.model.occ.addRectangle(
    0.0,
    0.0,
    z_half,
    Lx,
    Ly,
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Fragment glacier with internal planes
#
# This creates geometric edges at z = Lz/4 and z = Lz/2.
# -------------------------------------------------------------------------

domain, _ = gmsh.model.occ.fragment(
    domain,
    [
        (2, plane_quarter),
        (2, plane_half),
    ],
)

gmsh.model.occ.synchronize()

# -------------------------------------------------------------------------
# Get all geometric curves
# -------------------------------------------------------------------------

curves = gmsh.model.getEntities(1)

curve_tags = [
    tag
    for dim, tag in curves
]

# -------------------------------------------------------------------------
# Physical group for outline
# -------------------------------------------------------------------------

if curve_tags:

    gmsh.model.addPhysicalGroup(
        1,
        curve_tags,
        1,
    )

    gmsh.model.setPhysicalName(
        1,
        1,
        "OUTLINE",
    )

# -------------------------------------------------------------------------
# Generate ONLY 1D line mesh
# -------------------------------------------------------------------------

gmsh.model.mesh.generate(1)

# -------------------------------------------------------------------------
# Output
# -------------------------------------------------------------------------

os.makedirs(
    "11",
    exist_ok=True,
)

filename = "11/Lx1500_30C_S90_outline.msh"

gmsh.write(filename)

print(f"Generated: {filename}")

gmsh.finalize()

Info    : Meshing 1D...nts                                                                                                                
Info    : [  0%] Meshing curve 414 (Line)
Info    : [ 10%] Meshing curve 415 (Line)
Info    : [ 10%] Meshing curve 416 (Line)
Info    : [ 10%] Meshing curve 417 (Line)
Info    : [ 10%] Meshing curve 418 (Line)
Info    : [ 10%] Meshing curve 419 (Line)
Info    : [ 10%] Meshing curve 420 (Line)
Info    : [ 10%] Meshing curve 421 (Line)
Info    : [ 10%] Meshing curve 422 (Line)
Info    : [ 10%] Meshing curve 423 (Line)
Info    : [ 10%] Meshing curve 424 (Line)
Info    : [ 10%] Meshing curve 425 (Line)
Info    : [ 10%] Meshing curve 426 (Line)
Info    : [ 10%] Meshing curve 427 (Line)
Info    : [ 10%] Meshing curve 428 (Line)
Info    : [ 10%] Meshing curve 429 (Line)
Info    : [ 10%] Meshing curve 430 (Line)
Info    : [ 10%] Meshing curve 431 (Line)
Info    : [ 10%] Meshing curve 432 (Line)
Info    : [ 10%] Meshing curve 433 (Line)
Info    : [ 10%] Mesh

# Outline - XDMF

In [4]:
import meshio

mesh_files = ["11/Lx1500_30C_S90_outline.msh"]

for mesh_file in mesh_files:
    mesh = meshio.read(mesh_file)
    cells = mesh.get_cells_type("line")
    points = mesh.points

    meshio.write(
        mesh_file.replace(".msh", ".xdmf"),
        meshio.Mesh(
            points=points,
            cells={"line": cells},
        ),
    )
   